# CA1 – Apartment Price Prediction Using Machine Learning

**Module:** Machine Learning  
**Programme:** Higher Diploma in Data Analytics for Business  
**Student ID:** 2026328  
**Assignment:** CA1  
**Project Topic:** Apartment Price Prediction in Mongolia  
**Data Source:** [Unegui.mn](https://www.unegui.mn/l-hdlh/l-hdlh-zarna/)  
**GitHub Repository:** https://github.com/Dashkaa1114/Machine-Learning-Sep26

---

## 1. Introduction

Apartment prices can be influenced by several factors, including location, apartment size, number of rooms, and building age. Understanding these factors can help buyers, sellers, and property investors make better decisions.

This project aims to develop and evaluate Machine Learning models to predict apartment prices in Mongolia.

The dataset is collected from Unegui.mn, a Mongolian online classified advertisements platform. The collected information includes apartment prices, locations, sizes, number of rooms, building years, and apartment complex names.

## 2. Project Aim and Objectives

The main aim of this project is to predict apartment prices using Machine Learning regression models.

The objectives are to:

1. Examine and clean the collected apartment data.
2. Explore the relationships between apartment prices and property features.
3. Prepare the dataset for Machine Learning.
4. Train and evaluate at least two regression models.
5. Apply cross-validation and hyperparameter tuning.
6. Compare model performance and identify important features affecting apartment prices.

## 3. Methodology

This project follows the main stages of a Machine Learning workflow:

**Data Collection → Data Cleaning → Exploratory Data Analysis → Data Preprocessing → Model Training → Model Evaluation**

The target variable is `price_mnt`, which represents the advertised apartment price in Mongolian Tugrik (MNT).

The models will be evaluated using suitable regression metrics, such as MAE, RMSE, and R².

## 4. Expected Outcome

The expected outcome is to identify a suitable Machine Learning model for apartment price prediction and understand which apartment features have the strongest relationship with price.

The results will be based on advertised property prices rather than confirmed final sale prices.

In [ ]:
# Import libraries.
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
# Find the project folder.
folder = Path.cwd().resolve()

if folder.name.lower() == "notebooks":
    project_dir = folder.parent
else:
    project_dir = folder

# Load database settings.
load_dotenv(project_dir / ".env")

# Create PostgreSQL connection.
db_url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=int(os.getenv("PGPORT", "5432")),
    database=os.getenv("PGDATABASE"),
)

engine = create_engine(db_url, pool_pre_ping=True)

print("Database connection is ready.")

Database connection is ready.


In [3]:
# Load apartment data from PostgreSQL.

query = """SELECT * FROM public.apartment_listings_live"""

with engine.connect() as conn:
    conn.execute(text("SET TRANSACTION READ ONLY"))
    df_raw = pd.read_sql_query(text(query), conn)

print("Data loaded successfully.")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

display(df_raw.head())

Data loaded successfully.
Rows: 3174
Columns: 31


,listing_id,listing_url,title,price_text,price_mnt,currency,rooms,city_region,posted_text,posted_at,...,floor_number,payment_terms,window_count,construction_status,elevator,collected_at,complex_name,complex_source,complex_review_needed,published_at
0,10831343,https://www.unegui.mn/adv/10831343_bzd-26-r-kh...,"Бзд, 26-р хороо mandala town 83.09мк2 мастерта...",556.7 сая ₮,556700000,MNT,3.0,Баянзүрх,None,None,...,14,Бэлэн төлөлтөөр,3,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:27.378858+00:00,NaN,NaN,None,None
1,10797077,https://www.unegui.mn/adv/10797077_sbd-century...,"Сбд, сentury business apartment-д 87.93мкв мас...",800 сая ₮,800000000,MNT,3.0,Сүхбаатар,None,None,...,16,Банкны зээлээр,4,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:33.462158+00:00,NaN,NaN,None,None
2,10836867,https://www.unegui.mn/adv/10836867_khud-15-r-k...,"Худ, 15-р хороо хүннү 2222 1-р ээлж 55,04мкв 2...",445 сая ₮,445000000,MNT,2.0,Хан-Уул,None,None,...,11,Банкны зээлээр,5,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:39.501306+00:00,NaN,NaN,None,None
3,10824878,https://www.unegui.mn/adv/10824878_khud-18-r-k...,Худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,615 сая ₮,615000000,MNT,3.0,Хан-Уул,None,None,...,18,Банкны зээлээр,4,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:45.541792+00:00,NaN,NaN,None,None
4,10837139,https://www.unegui.mn/adv/10837139_khud-king-t...,"Худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",450 сая ₮,450000000,MNT,3.0,Хан-Уул,None,None,...,8,Банкны зээлээр,3,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:51.505214+00:00,NaN,NaN,None,None


In [43]:
df = df_raw.copy()

In [44]:
df.columns = df.columns.str.strip().str.lower()
print(df.columns.tolist())

['listing_id', 'listing_url', 'title', 'price_text', 'price_mnt', 'currency', 'rooms', 'city_region', 'posted_text', 'posted_at', 'description', 'latitude', 'longitude', 'floor_material', 'balcony', 'building_year', 'garage', 'window_type', 'total_floors', 'door_type', 'area_m2', 'floor_number', 'payment_terms', 'window_count', 'construction_status', 'elevator', 'collected_at', 'complex_name', 'complex_source', 'complex_review_needed', 'published_at']


In [45]:
# Rename target value 
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
df = df.rename(columns={"default_payment_next_month": "default"})

In [46]:
df.head()

,listing_id,listing_url,title,price_text,price_mnt,currency,rooms,city_region,posted_text,posted_at,...,floor_number,payment_terms,window_count,construction_status,elevator,collected_at,complex_name,complex_source,complex_review_needed,published_at
0,10831343,https://www.unegui.mn/adv/10831343_bzd-26-r-kh...,"Бзд, 26-р хороо mandala town 83.09мк2 мастерта...",556.7 сая ₮,556700000,MNT,3.0,Баянзүрх,None,None,...,14,Бэлэн төлөлтөөр,3,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:27.378858+00:00,NaN,NaN,None,None
1,10797077,https://www.unegui.mn/adv/10797077_sbd-century...,"Сбд, сentury business apartment-д 87.93мкв мас...",800 сая ₮,800000000,MNT,3.0,Сүхбаатар,None,None,...,16,Банкны зээлээр,4,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:33.462158+00:00,NaN,NaN,None,None
2,10836867,https://www.unegui.mn/adv/10836867_khud-15-r-k...,"Худ, 15-р хороо хүннү 2222 1-р ээлж 55,04мкв 2...",445 сая ₮,445000000,MNT,2.0,Хан-Уул,None,None,...,11,Банкны зээлээр,5,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:39.501306+00:00,NaN,NaN,None,None
3,10824878,https://www.unegui.mn/adv/10824878_khud-18-r-k...,Худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,615 сая ₮,615000000,MNT,3.0,Хан-Уул,None,None,...,18,Банкны зээлээр,4,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:45.541792+00:00,NaN,NaN,None,None
4,10837139,https://www.unegui.mn/adv/10837139_khud-king-t...,"Худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",450 сая ₮,450000000,MNT,3.0,Хан-Уул,None,None,...,8,Банкны зээлээр,3,Ашиглалтад орсон,Цахилгаан шаттай,2026-10-21 09:36:51.505214+00:00,NaN,NaN,None,None


In [47]:
df = df.map(lambda s: s.lower() if isinstance(s, str) else s)

In [48]:
df.head()

,listing_id,listing_url,title,price_text,price_mnt,currency,rooms,city_region,posted_text,posted_at,...,floor_number,payment_terms,window_count,construction_status,elevator,collected_at,complex_name,complex_source,complex_review_needed,published_at
0,10831343,https://www.unegui.mn/adv/10831343_bzd-26-r-kh...,"бзд, 26-р хороо mandala town 83.09мк2 мастерта...",556.7 сая ₮,556700000,mnt,3.0,баянзүрх,None,None,...,14,бэлэн төлөлтөөр,3,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:27.378858+00:00,NaN,NaN,None,None
1,10797077,https://www.unegui.mn/adv/10797077_sbd-century...,"сбд, сentury business apartment-д 87.93мкв мас...",800 сая ₮,800000000,mnt,3.0,сүхбаатар,None,None,...,16,банкны зээлээр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:33.462158+00:00,NaN,NaN,None,None
2,10836867,https://www.unegui.mn/adv/10836867_khud-15-r-k...,"худ, 15-р хороо хүннү 2222 1-р ээлж 55,04мкв 2...",445 сая ₮,445000000,mnt,2.0,хан-уул,None,None,...,11,банкны зээлээр,5,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:39.501306+00:00,NaN,NaN,None,None
3,10824878,https://www.unegui.mn/adv/10824878_khud-18-r-k...,худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,615 сая ₮,615000000,mnt,3.0,хан-уул,None,None,...,18,банкны зээлээр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:45.541792+00:00,NaN,NaN,None,None
4,10837139,https://www.unegui.mn/adv/10837139_khud-king-t...,"худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",450 сая ₮,450000000,mnt,3.0,хан-уул,None,None,...,8,банкны зээлээр,3,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:51.505214+00:00,NaN,NaN,None,None


In [49]:
import pandas as pd
from pathlib import Path

# Get the current working folder.
current_folder = Path.cwd()

# If we are inside "notebooks", use its parent folder.
if current_folder.name.lower() == "notebooks":
    project_folder = current_folder.parent
else:
    project_folder = current_folder

# Create the path to the Excel file.
data_folder = project_folder / "data"
excel_file = data_folder / "APARTMENT_LOC_PATTERNS.xlsx"

# Read the first sheet and load values as text.
patterns_df = pd.read_excel(
    excel_file,
    sheet_name=0,
    dtype=str,
    engine="openpyxl"
)

# Replace missing values with empty strings.
patterns_df = patterns_df.fillna("")

# Show the number of rows.
total_patterns = len(patterns_df)
print("Total patterns:", total_patterns)

# Show the first five rows.
display(patterns_df.head())

Total patterns: 254


,PATT_NAME,PATTERNS,PATTERNS1,DIST
0,санто апартмент,(санто хотхонд|санто хотхон-д|santo hothond|sa...,(санто хотхон|санто хотхон|santo hothon|santo ...,баянзүрх
1,ромео жульет,(ромео жульет хотхонд|ромео жульет хотхон-д|ро...,(ромео жульет хотхон|ромео жульет|ромео жульет...,баянзүрх
2,роял кастл,(роял кастл хотхонд|роял кастл хотхон-д|роял к...,(роял кастл хотхон|роял кастл),баянзүрх
3,сенатор апартмент,(сенатор апартмент хотхонд|сенатор апартмент х...,(сенатор апартмент хотхон|сенатор апартмент|се...,баянзүрх
4,хаппи таун,(хаппи таунд|хаппи таун-д|happy townd|happy to...,(хаппи таун|happy town),баянзүрх


In [50]:
# 1. Find the project folder.
project_dir = Path.cwd().resolve()

if project_dir.name.lower() == "notebooks":
    project_dir = project_dir.parent

# 2. Load apartment patterns from Excel.
pattern_path = project_dir / "data" / "APARTMENT_LOC_PATTERNS.xlsx"

patterns_df = pd.read_excel(
    pattern_path,
    sheet_name=0,
    dtype=str,
    engine="openpyxl"
).fillna("")

print("Total patterns:", len(patterns_df))
display(patterns_df.head())

Total patterns: 254


,PATT_NAME,PATTERNS,PATTERNS1,DIST
0,санто апартмент,(санто хотхонд|санто хотхон-д|santo hothond|sa...,(санто хотхон|санто хотхон|santo hothon|santo ...,баянзүрх
1,ромео жульет,(ромео жульет хотхонд|ромео жульет хотхон-д|ро...,(ромео жульет хотхон|ромео жульет|ромео жульет...,баянзүрх
2,роял кастл,(роял кастл хотхонд|роял кастл хотхон-д|роял к...,(роял кастл хотхон|роял кастл),баянзүрх
3,сенатор апартмент,(сенатор апартмент хотхонд|сенатор апартмент х...,(сенатор апартмент хотхон|сенатор апартмент|се...,баянзүрх
4,хаппи таун,(хаппи таунд|хаппи таун-д|happy townd|happy to...,(хаппи таун|happy town),баянзүрх


In [51]:
import re

# Remove spaces and line breaks from the Excel column names.
patterns_df.columns = patterns_df.columns.str.strip().str.lower()

# Create an empty list to store names and regex patterns.
pattern_list = []

# Read both pattern columns.
for column_name in ["patterns", "patterns1"]:

    # Read each row in the Excel data.
    for index, row in patterns_df.iterrows():

        complex_name = row["patt_name"].strip()
        pattern_text = row[column_name].strip()

        # Skip rows with an empty name or pattern.
        if complex_name == "" or pattern_text == "":
            continue

        # Prepare the pattern for searching.
        regex = re.compile(pattern_text, flags=re.IGNORECASE)

        # Store the complex name and its pattern together.
        pattern_list.append((complex_name, regex))

print("Prepared patterns:", len(pattern_list))

Prepared patterns: 508


In [52]:
def find_complex_name(row):
    title = ""
    description = ""

    if pd.notna(row["title"]):
        title = str(row["title"])

    if pd.notna(row["description"]):
        description = str(row["description"])

    listing_text = title + " " + description
    listing_text = re.sub(r"\s+", " ", listing_text).strip()

    for complex_name, regex in pattern_list:
        if regex.search(listing_text):
            return complex_name

    return pd.NA

In [53]:
# Find the complex name for each row.
# axis=1 means the function receives one row at a time.
complex_names = df.apply(find_complex_name, axis=1)

# Save the results in a new column.
df["complex_name_clean"] = complex_names

# Select the columns we want to check.
result_columns = ["title", "complex_name_clean"]
results = df[result_columns]

# Display the first 20 rows.
display(results.head(20))

# Count rows where a complex name was found.
matched_rows = df["complex_name_clean"].notna().sum()

# Count rows where no complex name was found.
unmatched_rows = df["complex_name_clean"].isna().sum()

print("Matched rows:", matched_rows)
print("Unmatched rows:", unmatched_rows)

,title,complex_name_clean
0,"бзд, 26-р хороо mandala town 83.09мк2 мастерта...",NaN
1,"сбд, сentury business apartment-д 87.93мкв мас...",NaN
2,"худ, 15-р хороо хүннү 2222 1-р ээлж 55,04мкв 2...",NaN
3,худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,академи хотхон ph-1
4,"худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",кинг тауэр
5,нисэхт буянт өргөө хотхонд 62м2 2 өрөө,NaN
6,king tower хотхонд шинэ дуплекс 152 м.кв 4 өрөө,кинг тауэр
7,1-р хороололд гал тогоо тусдаа 5 өрөө 100мкв байр,NaN
8,"бзд, халдвартад 45.87мкв 2 өрөө",NaN
9,"худ 18-р хороо хүннү-2222, 1-р ээлжинд 69.93 м...",NaN


Matched rows: 836
Unmatched rows: 2338


In [54]:
df.head()

,listing_id,listing_url,title,price_text,price_mnt,currency,rooms,city_region,posted_text,posted_at,...,payment_terms,window_count,construction_status,elevator,collected_at,complex_name,complex_source,complex_review_needed,published_at,complex_name_clean
0,10831343,https://www.unegui.mn/adv/10831343_bzd-26-r-kh...,"бзд, 26-р хороо mandala town 83.09мк2 мастерта...",556.7 сая ₮,556700000,mnt,3.0,баянзүрх,None,None,...,бэлэн төлөлтөөр,3,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:27.378858+00:00,NaN,NaN,None,None,NaN
1,10797077,https://www.unegui.mn/adv/10797077_sbd-century...,"сбд, сentury business apartment-д 87.93мкв мас...",800 сая ₮,800000000,mnt,3.0,сүхбаатар,None,None,...,банкны зээлээр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:33.462158+00:00,NaN,NaN,None,None,NaN
2,10836867,https://www.unegui.mn/adv/10836867_khud-15-r-k...,"худ, 15-р хороо хүннү 2222 1-р ээлж 55,04мкв 2...",445 сая ₮,445000000,mnt,2.0,хан-уул,None,None,...,банкны зээлээр,5,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:39.501306+00:00,NaN,NaN,None,None,NaN
3,10824878,https://www.unegui.mn/adv/10824878_khud-18-r-k...,худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,615 сая ₮,615000000,mnt,3.0,хан-уул,None,None,...,банкны зээлээр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:45.541792+00:00,NaN,NaN,None,None,академи хотхон ph-1
4,10837139,https://www.unegui.mn/adv/10837139_khud-king-t...,"худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",450 сая ₮,450000000,mnt,3.0,хан-уул,None,None,...,банкны зээлээр,3,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:51.505214+00:00,NaN,NaN,None,None,кинг тауэр


In [55]:
len(df)

3174

In [56]:
df["complex_name_clean"].isna().sum()

np.int64(2338)

In [57]:
df = df[df["complex_name_clean"].notna()]
df.shape

(836, 32)

In [58]:
df.head()

,listing_id,listing_url,title,price_text,price_mnt,currency,rooms,city_region,posted_text,posted_at,...,payment_terms,window_count,construction_status,elevator,collected_at,complex_name,complex_source,complex_review_needed,published_at,complex_name_clean
3,10824878,https://www.unegui.mn/adv/10824878_khud-18-r-k...,худ 18-р хороо академи 2 хотхонд 69.93 мкв 3 ө...,615 сая ₮,615000000,mnt,3.0,хан-уул,None,None,...,банкны зээлээр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:45.541792+00:00,NaN,NaN,None,None,академи хотхон ph-1
4,10837139,https://www.unegui.mn/adv/10837139_khud-king-t...,"худ, king tower 4-р ээлжинд 80м/кв 3 өрөө",450 сая ₮,450000000,mnt,3.0,хан-уул,None,None,...,банкны зээлээр,3,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:36:51.505214+00:00,NaN,NaN,None,None,кинг тауэр
6,10837133,https://www.unegui.mn/adv/10837133_king-tower-...,king tower хотхонд шинэ дуплекс 152 м.кв 4 өрөө,988 сая ₮,988000000,mnt,4.0,хан-уул,None,None,...,бэлэн төлөлтөөр,5,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:37:03.569148+00:00,NaN,NaN,None,None,кинг тауэр
11,10836874,https://www.unegui.mn/adv/10836874_river-garde...,river garden-2 120 мкв 4 өрөө байр,1.14 тэрбум ₮,1140000000,mnt,4.0,хан-уул,None,None,...,банкны зээлээр,5,ашиглалтад орсон,цахилгаан шаттай,2026-10-21 09:37:27.749042+00:00,NaN,NaN,None,None,ривер гарден 1
15,10676986,https://www.unegui.mn/adv/10676986_narny-guurn...,нарны гүүрний баруун талд алтай хотхонд 2 өрөө...,281 сая ₮,281000000,mnt,2.0,баянгол,None,None,...,бэлэн төлөлтөөр,4,ашиглалтад орсон,цахилгаан шаттай,2026-10-22 12:49:09.120278+00:00,гүүрний баруун талд алтай,title_and_description,True,None,алтай хотхон


In [59]:
df["complex_name_clean"].value_counts(dropna=False)

complex_name_clean
кинг тауэр             82
алтай хотхон           52
ривер гарден 1         30
жапан таун             29
цэнгэлдэх              24
                       ..
гарден резиденс         1
анома хотхон            1
дархан өргөө хотхон     1
euro town хотхон        1
поларис 1 апартмент     1
Name: count, Length: 150, dtype: int64

In [61]:
df.isna().sum()

listing_id                  0
listing_url                 0
title                       0
price_text                  0
price_mnt                   0
currency                    0
rooms                      47
city_region                84
posted_text              2060
posted_at                2060
description                 0
latitude                 2060
longitude                2060
floor_material              0
balcony                     0
building_year               0
garage                      0
window_type                 0
total_floors                0
door_type                   0
area_m2                     0
floor_number                0
payment_terms               0
window_count                0
construction_status         0
elevator                    0
collected_at                0
complex_name             1348
complex_source            554
complex_review_needed     554
published_at             1398
dtype: int64

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2060 entries, 0 to 2059
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   listing_id             2060 non-null   str                
 1   listing_url            2060 non-null   str                
 2   title                  2060 non-null   str                
 3   price_text             2060 non-null   str                
 4   price_mnt              2060 non-null   int64              
 5   currency               2060 non-null   str                
 6   rooms                  2013 non-null   float64            
 7   city_region            1976 non-null   str                
 8   posted_text            0 non-null      object             
 9   posted_at              0 non-null      object             
 10  description            2060 non-null   str                
 11  latitude               0 non-null      object             
 12  lon

In [63]:
df.duplicated().sum()

np.int64(0)

In [64]:
df.columns

Index(['listing_id', 'listing_url', 'title', 'price_text', 'price_mnt',
       'currency', 'rooms', 'city_region', 'posted_text', 'posted_at',
       'description', 'latitude', 'longitude', 'floor_material', 'balcony',
       'building_year', 'garage', 'window_type', 'total_floors', 'door_type',
       'area_m2', 'floor_number', 'payment_terms', 'window_count',
       'construction_status', 'elevator', 'collected_at', 'complex_name',
       'complex_source', 'complex_review_needed', 'published_at'],
      dtype='str')

In [65]:
df["city_region"].unique()

<StringArray>
[      'баянзүрх',      'сүхбаатар',        'хан-уул',      'чингэлтэй',
        'баянгол', 'сонгинохайрхан',          'орхон',              nan,
     'дархан-уул',            'төв',         'хэнтий']
Length: 11, dtype: str

In [66]:
df = df.drop(columns=["listing_id", "listing_url", "price_text","currency","posted_text", "posted_at",
                      "latitude","longitude"])

In [67]:
df.columns

Index(['price_mnt', 'rooms', 'city_region', 'description', 'floor_material',
       'balcony', 'building_year', 'garage', 'window_type', 'total_floors',
       'door_type', 'area_m2', 'floor_number', 'payment_terms', 'window_count',
       'construction_status', 'elevator', 'collected_at', 'complex_name',
       'complex_source', 'complex_review_needed', 'published_at'],
      dtype='str')

Total patterns: 254


,PATT_NAME,PATTERNS,PATTERNS1,DIST
0,санто апартмент,(санто хотхонд|санто хотхон-д|santo hothond|sa...,(санто хотхон|санто хотхон|santo hothon|santo ...,баянзүрх
1,ромео жульет,(ромео жульет хотхонд|ромео жульет хотхон-д|ро...,(ромео жульет хотхон|ромео жульет|ромео жульет...,баянзүрх
2,роял кастл,(роял кастл хотхонд|роял кастл хотхон-д|роял к...,(роял кастл хотхон|роял кастл),баянзүрх
3,сенатор апартмент,(сенатор апартмент хотхонд|сенатор апартмент х...,(сенатор апартмент хотхон|сенатор апартмент|се...,баянзүрх
4,хаппи таун,(хаппи таунд|хаппи таун-д|happy townd|happy to...,(хаппи таун|happy town),баянзүрх


Prepared rules: 254


In [46]:
display(df.describe())

,price_mnt,rooms,building_year,total_floors,area_m2,floor_number,window_count
count,2.060000e+03,2013.000000,2060.000000,2060.000000,2060.000000,2060.000000,2060.000000
mean,4.726986e+08,2.727769,2018.230097,13.448058,88.502073,7.689806,3.377670
std,5.076561e+08,0.960662,9.091915,4.797879,234.208428,4.773082,1.432329
min,6.800000e+07,1.000000,1980.000000,1.000000,1.000000,1.000000,1.000000
25%,2.100000e+08,2.000000,2015.000000,10.000000,52.000000,4.000000,2.000000
50%,3.100000e+08,3.000000,2021.000000,15.000000,68.410000,7.000000,3.000000
75%,5.400000e+08,3.000000,2025.000000,16.000000,94.000000,11.000000,4.000000
max,6.700000e+09,8.000000,2026.000000,25.000000,8629.000000,25.000000,8.000000


In [52]:
print('Avg price_mnt:', df.price_mnt.mean())
print('Min price_mnt:', df.price_mnt.min())
print('Max price_mnt:', df.price_mnt.max())

Avg sq_meter: 472698592.2330097
Min sq_meter: 68000000
Max sq_meter: 6700000000


In [53]:
print('Avg area_m2:', df.area_m2.mean())
print('Min area_m2:', df.area_m2.min())
print('Max area_m2:', df.area_m2.max())

Avg area_m2: 88.50207281553399
Min area_m2: 1.0
Max area_m2: 8629.0


In [20]:
print("\nMissing values:", int(df.isnull().sum().sum()))


Missing values: 8105


In [ ]:
df[["sex", "education", "marriage", "default"]].nunique().to_frame("Number of categories")